## 1. Silver Layer Setup & Quality Routing
This initial block configures the environment and establishes the central data governance engine for the Silver layer. 

* **Routing Engine:** Defines a reusable `route_and_save` function implementing the Kimball Error Event Schema. 
* **Execution Flow:** For every dataset, this function eliminates primary key duplicates, evaluates row-level data against strict quality conditions, writes the clean data to the Silver layer, and redirects any non-compliant records to a centralized `quarantine_master` table for auditing.

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "dbw_proman_lakehouse_01"
SCHEMA = "default"
TARGET_PATH = f"{CATALOG}.{SCHEMA}"

# Reusable routing helper: saves clean rows to Silver and bad rows to Quarantine
def route_and_save(df_clean, entity_name, id_col, is_valid_condition, rejection_expression):
    df_clean = df_clean.dropDuplicates([id_col])
    df_valid = df_clean.filter(is_valid_condition)
    df_invalid = df_clean.filter(~is_valid_condition)

    (
        df_valid.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TARGET_PATH}.silver_{entity_name}")
    )

    df_quarantine_rows = df_invalid.select(
        F.lit(f"bronze_{entity_name}").alias("source_table"),
        F.coalesce(F.col(id_col).cast("string"), F.lit("UNKNOWN")).alias("record_id"),
        rejection_expression.alias("issue_reason"),
        F.current_timestamp().alias("quarantined_at"),
    )

    (
    df_quarantine_rows.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"source_table = 'bronze_{entity_name}'")
        .saveAsTable(f"{TARGET_PATH}.quarantine_master")
    )

    print(f"Processed '{entity_name}': {df_valid.count()} passed, {df_invalid.count()} quarantined.")

### 2. Conform & Validate Dimensions: Users
Processes the operational user data with a focus on data conforming and financial modeling:

* **String Conforming:** Trims whitespace, lowercases emails, and standardizes `full_name` using title case (`initcap`).
* **Financial Fallback:** Implements graceful fallback logic. If an employee lacks a defined `hourly_rate`, a default baseline of €75.0 is automatically applied to ensure downstream financial aggregations do not fail.

In [0]:
has_hourly_rate = "hourly_rate" in spark.table(f"{TARGET_PATH}.bronze_users").columns

df_users = (
    spark.table(f"{TARGET_PATH}.bronze_users")
    .select(
        F.col("user_id").cast("int").alias("user_id"),
        F.initcap(F.trim(F.col("full_name"))).alias("full_name"),
        F.lower(F.trim(F.col("email"))).alias("email"),
        F.trim(F.col("department")).alias("department"),
        F.trim(F.col("role")).alias("role"),
        (F.col("hourly_rate").cast("double") if has_hourly_rate else F.lit(75.0)).alias("hourly_rate"),
        F.col("is_active").cast("boolean").alias("is_active"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_users = (
    F.col("user_id").isNotNull() 
    & F.col("email").contains("@")
    & (F.col("hourly_rate") > 0.0)
)

reason_users = F.concat_ws(
    "; ",
    F.when(F.col("user_id").isNull(), "Null user_id"),
    F.when(~F.col("email").contains("@"), "Invalid email format"),
    F.when(F.col("hourly_rate") <= 0.0, "Non-positive hourly rate"),
)

route_and_save(df_users, "users", "user_id", valid_users, reason_users)

Processed 'users': 20 passed, 1 quarantined.


### 3. Conform & Validate Dimensions: Projects
Standardizes core project attributes with strict logical and financial validation:

* **Temporal Boundaries:** Ensures a project's `start_date` does not occur after its `due_date`.
* **Financial Integrity:** Guarantees that neither hour-based (`budget_hours`) nor capital budgets (`budget_amount`) contain negative values.

In [0]:
has_budget_amount = "budget_amount" in spark.table(f"{TARGET_PATH}.bronze_projects").columns

df_projects = (
    spark.table(f"{TARGET_PATH}.bronze_projects")
    .select(
        F.trim(F.col("project_id")).alias("project_id"),
        F.trim(F.col("project_name")).alias("project_name"),
        F.trim(F.col("status")).alias("status"),
        F.to_date(F.trim(F.col("start_date"))).alias("start_date"),
        F.to_date(F.trim(F.col("due_date"))).alias("due_date"),
        F.col("budget_hours").cast("double").alias("budget_hours"),
        (F.col("budget_amount").cast("double") if has_budget_amount else (F.col("budget_hours") * 75.0)).alias("budget_amount"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_projects = (
    F.col("project_id").isNotNull()
    & (F.col("start_date") <= F.col("due_date"))
    & (F.col("budget_hours") >= 0)
    & (F.col("budget_amount") >= 0)
)

reason_projects = F.concat_ws(
    "; ",
    F.when(F.col("project_id").isNull(), "Null project_id"),
    F.when(F.col("start_date") > F.col("due_date"), "start_date exceeds due_date"),
    F.when(F.col("budget_hours") < 0, "Negative budget_hours"),
    F.when(F.col("budget_amount") < 0, "Negative budget_amount"),
)

route_and_save(df_projects, "projects", "project_id", valid_projects, reason_projects)

Processed 'projects': 7 passed, 1 quarantined.


### 4. Conform & Validate Dimensions: Tasks
Processes granular operational tasks to ensure accurate Agile metrics and cumulative flow diagrams in the Gold layer:

* **Domain Constraints:** Strictly enforces domain constraints on the `status` column. 
* **Quarantine Enforcement:** Any task containing an unrecognized status outside the allowed array (e.g., "To Do", "In Progress", "Done", "Blocked") is automatically quarantined to prevent corrupting executive dashboard workflows.

In [0]:
allowed_task_statuses = ["To Do", "In Progress", "Done", "Blocked"]

df_tasks = (
    spark.table(f"{TARGET_PATH}.bronze_tasks")
    .select(
        F.trim(F.col("task_id")).alias("task_id"),
        F.trim(F.col("project_id")).alias("project_id"),
        F.col("assigned_user_id").cast("int").alias("assigned_user_id"),
        F.trim(F.col("task_name")).alias("task_name"),
        F.trim(F.col("priority")).alias("priority"),
        F.trim(F.col("status")).alias("status"),
        F.col("estimated_hours").cast("int").alias("estimated_hours"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_tasks = (
    F.col("task_id").isNotNull()
    & F.col("project_id").isNotNull()
    & (F.col("estimated_hours") > 0)
    & F.col("status").isin(allowed_task_statuses)
)

reason_tasks = F.concat_ws(
    "; ",
    F.when(F.col("task_id").isNull(), "Null task_id"),
    F.when(F.col("project_id").isNull(), "Missing project_id foreign key"),
    F.when(F.col("estimated_hours") <= 0, "estimated_hours must be positive"),
    F.when(~F.col("status").isin(allowed_task_statuses), "Invalid task status"),
)

route_and_save(df_tasks, "tasks", "task_id", valid_tasks, reason_tasks)

Processed 'tasks': 34 passed, 1 quarantined.


### 5. Process Transactional Facts: Time-Tracking
Applies a "Schema-on-Read" pattern to parse and flatten the nested JSON time-tracking payloads into a tabular event log. Because these represent immutable human labor entries, the validation logic is highly strict:

* **Schema-on-Read Parsing:** The Bronze layer safely ingests nested arrays as raw strings to prevent pipeline crashes from upstream schema drift. Here, `from_json` applies a strict DDL schema to parse the string back into a structured array before using PySpark's `explode` function.
* **Bound Checks:** Guards against impossible bounds, such as logging more than 24 hours (`hours_worked`) in a single day or claiming over 100% `progress_pct`.
* **Temporal Checks:** Prevents future-dated work entries.

In [0]:
# Define the expected nested structure 
logs_schema = "array<struct<entry_id:string, project_id:string, task_id:string, user_id:int, date:string, hours_worked:double, progress_pct:double, activity_description:string, is_billable:boolean>>"

df_timetracking = (
    spark.table(f"{TARGET_PATH}.bronze_timetracking")
    .withColumn("logs_parsed", F.from_json(F.col("logs"), logs_schema))
    .select(F.explode("logs_parsed").alias("log"), F.col("_source_file"))
    .select(
        F.trim(F.col("log.entry_id")).alias("entry_id"),
        F.trim(F.col("log.project_id")).alias("project_id"),
        F.trim(F.col("log.task_id")).alias("task_id"),
        F.col("log.user_id").cast("int").alias("user_id"),
        F.to_date(F.trim(F.col("log.date"))).alias("work_date"),
        F.col("log.hours_worked").cast("double").alias("hours_worked"),
        F.coalesce(F.col("log.progress_pct").cast("double"), F.lit(0.0)).alias("progress_pct"),
        F.trim(F.col("log.activity_description")).alias("activity_description"),
        F.col("_source_file"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_timetracking = (
    F.col("entry_id").isNotNull()
    & F.col("task_id").isNotNull()
    & (F.col("task_id") != "")
    & (F.col("hours_worked") > 0.0)
    & (F.col("hours_worked") <= 24.0)
    & (F.col("work_date") <= F.current_date())
    & (F.col("progress_pct") >= 0.0)
    & (F.col("progress_pct") <= 100.0)
)

reason_timetracking = F.concat_ws(
    "; ",
    F.when(F.col("entry_id").isNull(), "Null entry_id"),
    F.when(F.col("task_id").isNull() | (F.col("task_id") == ""), "Missing task_id foreign key"),
    F.when(
        (F.col("hours_worked") <= 0.0) | (F.col("hours_worked") > 24.0),
        "hours_worked must be between 0 and 24",
    ),
    F.when(F.col("work_date") > F.current_date(), "Future work_date"),
    F.when(
        (F.col("progress_pct") < 0.0) | (F.col("progress_pct") > 100.0),
        "progress_pct must be between 0 and 100",
    ),
)

route_and_save(
    df_timetracking, "timetracking", "entry_id", valid_timetracking, reason_timetracking
)

Processed 'timetracking': 102 passed, 9 quarantined.


### 6. Process Transactional Facts: Documents
Filters the raw document manifest to isolate project deliverables and artifacts (excluding vendor invoices):

* **Categorical Grouping:** Standardizes file extensions to uppercase for easier categorical grouping.
* **Storage Analytics:** Performs a conversion of file size from raw bytes into Megabytes (`file_size_mb`), enabling direct storage footprint analytics in the Gold layer.

In [0]:
df_documents = (
    spark.table(f"{TARGET_PATH}.bronze_documents_manifest")
    .filter(F.col("document_category") != "vendor_invoice")
    .select(
        F.trim(F.col("document_id")).alias("document_id"),
        F.trim(F.col("project_id")).alias("project_id"),
        F.trim(F.col("task_id")).alias("task_id"),
        F.trim(F.col("document_category")).alias("document_category"),
        F.trim(F.col("file_name")).alias("file_name"),
        F.upper(F.trim(F.col("file_extension"))).alias("file_type"),
        F.round(F.col("file_size_bytes") / (1024.0 * 1024.0), 2).alias("file_size_mb"),
        F.col("adls_raw_uri"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_documents = (
    F.col("document_id").isNotNull()
    & F.col("project_id").isNotNull()
    & (F.col("project_id") != "")
    & (F.col("file_size_mb") > 0.0)
)

reason_documents = F.concat_ws(
    "; ",
    F.when(F.col("document_id").isNull(), "Null document_id"),
    F.when(F.col("project_id").isNull() | (F.col("project_id") == ""), "Missing project_id link"),
    F.when(F.col("file_size_mb") <= 0.0, "Invalid file size"),
)

route_and_save(
    df_documents, "documents", "document_id", valid_documents, reason_documents
)

Processed 'documents': 46 passed, 3 quarantined.


### 7. Process Transactional Facts: Expenses
Specifically processes external vendor invoices for downstream financial aggregation:

* **Data Conforming:** Forces `vendor_name` into a standardized title case format to ensure clean aggregation by vendor in the Gold analytical layer.
* **Capital Validation:** Strictly validates that all parsed expense amounts represent positive capital values before allowing them to merge into project budgets.

In [0]:
df_expenses = (
    spark.table(f"{TARGET_PATH}.bronze_documents_manifest")
    .filter(F.col("document_category") == "vendor_invoice")
    .select(
        F.trim(F.col("expense_id")).alias("expense_id"),
        F.trim(F.col("project_id")).alias("project_id"),
        F.to_date(F.trim(F.col("expense_date"))).alias("expense_date"),
        F.trim(F.col("category")).alias("category"),
        F.initcap(F.trim(F.col("vendor_name"))).alias("vendor_name"),
        F.col("amount").cast("double").alias("amount"),
        F.col("adls_raw_uri"),
        F.current_timestamp().alias("_transformed_at"),
    )
)

valid_expenses = (
    F.col("expense_id").isNotNull()
    & F.col("project_id").isNotNull()
    & (F.col("amount") > 0.0)
)

reason_expenses = F.concat_ws(
    "; ",
    F.when(F.col("expense_id").isNull(), "Null expense_id"),
    F.when(F.col("project_id").isNull(), "Missing project_id foreign key"),
    F.when(F.col("amount") <= 0.0, "Amount must be greater than zero"),
)

route_and_save(
    df_expenses, "expenses", "expense_id", valid_expenses, reason_expenses
)

Processed 'expenses': 14 passed, 0 quarantined.
